In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot  as plt
from PIL import Image

In [ ]:
# Sniff device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # For Apple Silicon (M1/M2/M3/M4 Macs)
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


# Loss Modules
class ContentLoss(nn.Module):
    def __init__(self, feat_map: torch.Tensor):
        super().__init__()
        self.target = feat_map.detach()  # Detach from the computation graph

    def forward(self, x: torch.Tensor):
        # mes_loss(input, target)
        # element-wise mean squared error
        self.loss = nn.functional.mse_loss(x, self.target)
        return x


def gram_matrix(x: torch.Tensor):
    N, C, H, W = x.size()
    feat_map = x.view(N, C, H * W)
    # {N,C,Z} x {N,Z,C} = {N,C,C}
    G = torch.bmm(feat_map, feat_map.transpose(1, 2))  # Batch matrix multiplication
    # remove scaling effect from N, C, and H*W
    return G.div(N * C * H * W)


class StyleLoss(nn.Module):
    def __init__(self, feat_map: torch.Tensor):
        super().__init__()
        self.target = gram_matrix(
            feat_map
        ).detach()  # Detach from the computation graph

    def forward(self, x: torch.Tensor):
        G = gram_matrix(x)
        self.loss = nn.functional.mse_loss(G, self.target)
        return x


class Normalization(nn.Module):
    def __init__(self, mean: torch.tensor, std: torch.tensor):
        super().__init__()
        # reshape (n,) into (n,1,1) for broadcasting
        # align channel dimension
        self.mean = mean.detach().clone().view(-1, 1, 1)
        self.std = std.detach().clone().view(-1, 1, 1)

    def forward(self, x):
        return (x - self.mean) / self.std


# Define which layers to monitor for content and style
content_layers_default = ["conv_4"]
style_layers_default = ["conv_1", "conv_2", "conv_3", "conv_4", "conv_5"]


def assemble_model(
    vgg,
    normalization_mean,
    normalization_std,
    style_img,
    content_img,
    content_layers=content_layers_default,
    style_layers=style_layers_default,
):
    normalization = Normalization(normalization_mean, normalization_std).to(device)

    content_losses = []
    style_losses = []

    model = nn.Sequential(normalization)

    i = 0
    for layer in vgg.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f"conv_{i}"
        elif isinstance(layer, nn.ReLU):
            name = f"relu_{i}"
            # Inplace versions don't play nice with Content/Style loss
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f"pool_{i}"
        elif isinstance(layer, nn.BatchNorm2d):
            name = f"bn_{i}"
        else:
            raise RuntimeError(f"Unrecognized layer: {layer.__class__.__name__}")

        model.add_module(name, layer)

        if name in content_layers:
            target_feat = model(content_img).detach()
            content_loss = ContentLoss(target_feat)
            model.add_module(f"content_loss_{i}", content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            target_feat = model(style_img).detach()
            style_loss = StyleLoss(target_feat)
            model.add_module(f"style_loss_{i}", style_loss)
            style_losses.append(style_loss)

    # seek to layer idx before the last ContentLoss or StyleLoss
    # after which is FC layers
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break
    model = model[: (i + 1)]

    return model, style_losses, content_losses


def gradient_descent_loop(
    vgg,
    normalization_mean,
    normalization_std,
    content_img,
    style_img,
    input_img,
    num_steps=300,
    style_weight=1000000,
    content_weight=1,
):
    model, style_losses, content_losses = assemble_model(
        vgg,
        normalization_mean,
        normalization_std,
        style_img,
        content_img,
    )

    # We optimize the input image, not the model parameters
    input_img.requires_grad_(True)
    model.requires_grad_(False)

    # Limited-memory BFGS
    # See NST_Analysis > L-BFGS
    optimizer = optim.LBFGS([input_img])

    run = 0
    while run <= num_steps:

        def closure():
            nonlocal run

            # Clamp the image values to be between 0 and 1
            with torch.no_grad():
                input_img.clamp_(0, 1)

            optimizer.zero_grad()
            model(input_img)

            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            loss = style_weight * style_score + content_weight * content_score
            loss.backward()

            run += 1
            if run % 50 == 0:
                print(f"run {run}:")
                print(
                    f"Style Loss: {style_score.item():4f} |",
                    f"Content Loss: {content_score.item():4f}",
                )
                print("-" * 20)

            return style_score + content_score

        optimizer.step(closure)

    # Final clamp to guarantee pixel values stay valid
    with torch.no_grad():
        input_img.clamp_(0, 1)

    return input_img

In [ ]:
# Load & Set Values

# Load VGG-19 and Build the Model
# We use the features block and freeze the weights
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()

# Normalization constants for VGG
vgg_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
vgg_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

# Image transformation pipeline
imsize = 512
preprocess_img = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor()
])

# display imag
# expect a {C, H, W} tensor
def display_img(data: torch.Tensor):
    if len(data.shape) == 4:
        # if {N, C, H, W}, pick first
        data = data[0]
    elif len(data.shape) != 3:
        raise ValueError("Expected a 3D tensor with shape {C, H, W} or 4D tensor with shape {N, C, H, W}")
    # reshape into {N, H, W, C}
    rgb_arr = data.permute(1, 2, 0).cpu().numpy()
    plt.tight_layout()
    plt.axis('off')
    plt.imshow(rgb_arr)

# return a {1, 3, 512, 512} tensor
def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = preprocess_img(image).unsqueeze(0)  # Add batch dimension
    return image.to(device, torch.float)

imgC_path = "./assets/NST/C.jpg"
imgS_path = "./assets/NST/S2.jpg"

imgC = load_image(imgC_path)
imgS = load_image(imgS_path)

In [ ]:
print("Content Image")
display_img(imgC)

In [ ]:
print("Style Image")
display_img(imgS)

In [ ]:
styled_output = gradient_descent_loop(
    vgg, vgg_normalization_mean, vgg_normalization_std,
    imgC, imgS, imgC.clone(),  # Start with the content image as the initial input
    num_steps=300)

In [ ]:
print("Styled Output")
display_img(styled_output.cpu().detach())